# Perguntas e respostas sobre a Diretriz Brasileira de Hipertensão Arterial 2025: o impacto do RAG em modelos de linguagem abertos

**Disciplina:** Tópicos Avançados em Ciência de Dados (PLN) – PPGTI/IFPB – 2026.2
**Discente:** Rodrigo de Queiroz Gonçalves Velez
**Docentes:** Damires Souza e Alex Cunha

> **Status deste notebook:** esqueleto. Cada etapa tem uma célula de texto explicando o que será feito e uma célula de código ainda vazia. As células serão preenchidas aos poucos, etapa por etapa.

## Visão geral

### O problema
Uma pessoa leiga com dúvida sobre pressão alta pode perguntar a um modelo de linguagem (LLM). Os modelos abertos e pequenos, que rodam num computador comum, foram treinados **antes** da Diretriz Brasileira de Hipertensão Arterial 2025, que mudou a classificação da pressão. Então eles podem responder com critérios antigos.

### A pergunta de pesquisa
O **RAG** (*Retrieval-Augmented Generation*, geração aumentada por recuperação) busca na diretriz os trechos que respondem à pergunta e os entrega ao modelo junto com a pergunta. **Quanto isso melhora as respostas?**

- **P1:** sem RAG, quanto os modelos acertam e quanto se apoiam no critério antigo (diretriz de 2020)?
- **P2:** quanto o RAG aumenta a acurácia?
- **P3:** quando o RAG erra, a falha é da busca ou da geração?

### O que se compara
| Condição | Modelo | Recebe |
|---|---|---|
| Sem RAG (baseline) | Qwen 2.5 7B e Gemma 3 4B | Só a pergunta |
| Com RAG | Os mesmos modelos | A pergunta + os 4 trechos mais parecidos da diretriz |

O baseline é o **mesmo modelo sem RAG**. Assim, a única diferença entre as condições é o RAG, e qualquer ganho pode ser atribuído a ele.

### O caminho dos dados (pipeline)

```
Diretriz 2025 (PDF)
      │
      ├──► [Etapa 0] Conjunto de teste: 50 perguntas com resposta de referência (já feito, fora do notebook)
      │
      └──► [1] Texto numerado ──► [2] Limpeza e trechos ──► [3] Embeddings e índice ──► [4] Busca (top-4)
                                                                                          │
   50 perguntas ──► [5] Respostas SEM RAG ◄──────────────────────────────────────────────┤
                └─► [6] Respostas COM RAG ◄──────────────────────────────────────────────┘
                                │
                                ▼
                    [7] LLM-juiz ──► [8] Conferência humana (kappa) ──► [9] Estatística (acurácia, McNemar)
                                                                              │
                                                                              ▼
                                                              [10] Análise de erros ──► [11] Resultados
```

Cada etapa abaixo segue o mesmo formato: **o que faz, por que, entrada, saída**.

## 0. Preparação do ambiente

### 0.1 Instalação das bibliotecas
**O que faz:** instala as bibliotecas Python que o notebook usa (leitura de dados, cálculo com vetores, estatística e acesso aos modelos).
**Por que:** o notebook precisa rodar igual no Google Colab e no computador do pesquisador. As versões ficam fixadas para que o resultado possa ser reproduzido.

In [ ]:
# 0.1 Instalação das bibliotecas (a implementar)

### 0.2 Importações e parâmetros do experimento
**O que faz:** importa as bibliotecas e reúne, num só lugar, todos os parâmetros que podem mudar: tamanho dos trechos, sobreposição, quantidade de trechos recuperados (*top-k*), nomes dos modelos e temperatura.
**Por que:** com os parâmetros juntos, fica fácil saber exatamente como cada resultado foi produzido e repetir o experimento.

| Parâmetro | Valor previsto |
|---|---|
| Tamanho do trecho | ~1.000 caracteres |
| Sobreposição | ~200 caracteres |
| Trechos recuperados (*k*) | 4 |
| Modelo de embeddings | bge-m3 |
| Modelos avaliados | Qwen 2.5 7B, Gemma 3 4B |
| Temperatura | 0 (respostas determinísticas) |
| Juiz | Claude Opus 5.5 (via API) |

In [ ]:
# 0.2 Importações e parâmetros (a implementar)

### 0.3 Pastas e arquivos
**O que faz:** define onde estão os dados de entrada e onde serão gravados os resultados de cada etapa.
**Por que:** cada etapa grava o que produziu. Assim, uma etapa demorada (como gerar respostas) não precisa ser refeita quando só uma etapa posterior muda.

In [ ]:
# 0.3 Pastas e arquivos (a implementar)

### 0.4 Conexão com os modelos locais (Ollama)
**O que faz:** confere se o Ollama está rodando e se os três modelos estão disponíveis: `qwen2.5:7b`, `gemma3:4b` e `bge-m3`.
**Por que:** os modelos rodam localmente, sem custo e sem enviar dados para fora. Se algum faltar, o notebook avisa antes de começar.

In [ ]:
# 0.4 Verificação do Ollama e dos modelos (a implementar)

## 1. Dados de entrada

### 1.1 A diretriz numerada
**O que faz:** carrega o texto da Diretriz 2025 já extraído do PDF, com cada parágrafo, tabela e linha numerados por página (`data/processado/diretriz_2025_estrutura.jsonl`).
**Por que:** a numeração é o "endereço" de cada informação. É ela que liga a resposta de referência de cada pergunta ao lugar exato da diretriz.
**Origem:** produzido pelo script `src/gerar_perguntas.py` (etapa 1, sem IA), com conferência de integridade: o total de letras e números do PDF é igual ao extraído.

In [ ]:
# 1.1 Carregar a diretriz numerada (a implementar)

### 1.2 As 50 perguntas de teste
**O que faz:** carrega as 50 perguntas selecionadas (10 por tema), com a resposta de referência literal, a versão leiga e a localização na diretriz.
**Por que:** é o gabarito do experimento. As perguntas foram geradas pelo Claude Opus 5.5 a partir da diretriz, conferidas automaticamente no PDF, selecionadas pelo pesquisador e validadas por especialista em saúde.
**Origem:** `data/perguntas/` e `data/perguntas/selecao.json`. A geração (etapa 0) não é refeita aqui, porque tem custo de API e não é determinística.

In [ ]:
# 1.2 Carregar as perguntas selecionadas (a implementar)

### 1.3 Resumo dos dados
**O que faz:** mostra números básicos: páginas e parágrafos da diretriz, perguntas por tema e quantas tratam da nova classificação de 2025.
**Por que:** o artigo pede um resumo do corpus e do conjunto de teste. Essa célula gera esses números.

In [ ]:
# 1.3 Resumo dos dados (a implementar)

## 2. Limpeza e divisão em trechos (*chunking*)

### 2.1 Limpeza
**O que faz:** mantém só o corpo da diretriz (páginas 23 a 134) e remove capa, autores, sumário, declarações de interesse e referências bibliográficas.
**Por que:** essas partes não respondem às perguntas e só atrapalhariam a busca. As referências, por exemplo, contêm palavras como "hipertensão" centenas de vezes.

In [ ]:
# 2.1 Limpeza (a implementar)

### 2.2 Divisão em trechos
**O que faz:** junta parágrafos seguidos da mesma seção em trechos de cerca de 1.000 caracteres, com 200 de sobreposição. Cada tabela vira um trecho próprio. Cada trecho começa com o nome da seção.
**Por que:** a busca funciona melhor com blocos que têm uma ideia completa. Um fragmento de três palavras não tem sentido sozinho, e um capítulo inteiro mistura assuntos demais. A sobreposição evita que uma informação seja cortada ao meio.
**Saída:** `trechos.jsonl`, em que cada trecho guarda quais parágrafos e tabelas da diretriz ele contém.

In [ ]:
# 2.2 Divisão em trechos (a implementar)

### 2.3 Conferência dos trechos
**O que faz:** verifica se a evidência de cada uma das 50 perguntas está em pelo menos um trecho e mostra exemplos de trechos.
**Por que:** se a resposta de uma pergunta ficou de fora dos trechos, o RAG nunca vai encontrá-la. Melhor descobrir isso antes da busca.

In [ ]:
# 2.3 Conferência dos trechos (a implementar)

## 3. Embeddings e índice vetorial

### 3.1 Gerar os embeddings dos trechos
**O que faz:** transforma cada trecho num vetor de 1.024 números com o modelo **bge-m3**, que é multilíngue e entende português.
**Por que:** textos com sentidos parecidos viram vetores próximos. É isso que permite achar o trecho certo mesmo quando a pergunta usa palavras do dia a dia ("pressão de cima") e a diretriz usa termos técnicos ("PAS").
**Observação:** gerar embeddings **não é treinar** o modelo. O bge-m3 só "lê" os trechos. Os LLMs avaliados também não são alterados.

In [ ]:
# 3.1 Embeddings dos trechos (a implementar)

### 3.2 Montar e salvar o índice
**O que faz:** guarda todos os vetores para a busca e salva em arquivo.
**Por que:** gerar os embeddings leva alguns minutos. Com o índice salvo, as próximas execuções só carregam o arquivo.
**Decisão em aberto:** índice em **numpy** (funciona no Colab, sem Docker) ou em **pgvector** (como previsto na proposta). Com algumas centenas de trechos, os dois fazem a mesma busca exata.

In [ ]:
# 3.2 Índice vetorial (a implementar)

## 4. Busca e avaliação da recuperação

### 4.1 Função de busca
**O que faz:** recebe uma pergunta, gera o vetor dela com o bge-m3 e devolve os 4 trechos mais parecidos, pela **similaridade de cosseno**.
**Por que:** é a parte "R" (*retrieval*) do RAG. Os trechos que ela devolve são o contexto que o modelo vai receber.

In [ ]:
# 4.1 Função buscar(pergunta, k=4) (a implementar)

### 4.2 Recall@4
**O que faz:** para cada uma das 50 perguntas, verifica se o trecho com a resposta certa está entre os 4 devolvidos pela busca. O **Recall@4** é a proporção de perguntas em que isso acontece.
**Por que:** mede a busca isoladamente, antes de envolver o LLM. Se o trecho certo não chega, o modelo não tem como acertar com base na diretriz. Isso responde a parte da **P3**.

In [ ]:
# 4.2 Cálculo do Recall@4 (a implementar)

## 5. Respostas sem RAG (baseline)

### 5.1 Prompt sem RAG
**O que faz:** define o texto enviado ao modelo: uma instrução curta e a pergunta, sem nenhum trecho da diretriz.
**Por que:** mostra o que o modelo "sabe" sozinho, a partir do seu treinamento. O texto completo do prompt será apresentado no artigo, como pede a diretriz da disciplina.

In [ ]:
# 5.1 Prompt sem RAG (a implementar)

### 5.2 Gerar as respostas sem RAG
**O que faz:** envia as 50 perguntas a cada modelo (Qwen e Gemma), com temperatura 0, e grava as respostas e o tempo de resposta.
**Por que:** temperatura 0 faz o modelo sempre escolher a palavra mais provável, deixando o resultado repetível. O tempo de resposta (latência) é uma métrica secundária.
**Saída:** 100 respostas (50 perguntas × 2 modelos).

In [ ]:
# 5.2 Respostas sem RAG (a implementar)

## 6. Respostas com RAG

### 6.1 Prompt com RAG
**O que faz:** define o texto enviado ao modelo: instrução, os 4 trechos recuperados como contexto e a pergunta. A instrução pede para responder com base nos trechos, em linguagem simples.
**Por que:** é a parte "AG" (*augmented generation*) do RAG. A única diferença para o prompt sem RAG é o contexto, para que a comparação seja justa.

In [ ]:
# 6.1 Prompt com RAG (a implementar)

### 6.2 Gerar as respostas com RAG
**O que faz:** para cada pergunta, chama a busca, monta o prompt com os trechos e envia a cada modelo. Grava a resposta, os trechos usados e o tempo.
**Saída:** mais 100 respostas, totalizando 200 no experimento.

In [ ]:
# 6.2 Respostas com RAG (a implementar)

## 7. Avaliação com LLM-juiz (*LLM-as-a-Judge*)

### 7.1 Protocolo e prompt do juiz
**O que faz:** define como o juiz (Claude Opus 5.5) avalia cada resposta. Ele recebe a pergunta, a resposta de referência literal e a resposta do modelo, e devolve uma nota e uma justificativa.
**Critérios previstos:**
- **Correção** (principal): a resposta está de acordo com a referência? (1 = correta, 0 = incorreta)
- **Uso do critério antigo:** a resposta usa a classificação de 2020? (para a **P1**)
- **Fidelidade** (só com RAG): a resposta se apoia nos trechos recuperados, sem inventar?

**Por que:** avaliar 200 respostas à mão seria demorado. O juiz é de outra família de modelos, para evitar que favoreça respostas parecidas com as suas.

In [ ]:
# 7.1 Prompt e critérios do juiz (a implementar)

### 7.2 Executar o juiz
**O que faz:** envia as 200 respostas ao juiz pela API, grava as notas, as justificativas e o custo.
**Por que:** o custo é registrado como métrica secundária e para controle do orçamento do projeto.

In [ ]:
# 7.2 Execução do juiz (a implementar)

## 8. Conferência humana do juiz (kappa de Cohen)

**O que faz:** o pesquisador avalia à mão uma amostra de 25% das respostas, sem ver a nota do juiz. Depois, calcula o **kappa de Cohen** entre as duas avaliações.
**Por que:** o kappa mede o quanto juiz e humano concordam, descontando a concordância que aconteceria por acaso. Um kappa alto indica que dá para confiar no juiz nas demais respostas.

In [ ]:
# 8. Amostra para avaliação humana e cálculo do kappa (a implementar)

## 9. Análise estatística

### 9.1 Acurácia por modelo e condição
**O que faz:** calcula a proporção de respostas corretas para cada combinação (2 modelos × sem/com RAG), no geral e por tema.
**Por que:** responde à **P2** (quanto o RAG melhora) e mostra em que temas o ganho é maior.

In [ ]:
# 9.1 Acurácia (a implementar)

### 9.2 Teste de McNemar
**O que faz:** para cada modelo, compara as mesmas 50 perguntas sem e com RAG e verifica se a diferença de acertos é estatisticamente significativa.
**Por que:** como as perguntas são as mesmas nas duas condições (dados pareados), o McNemar olha só para as perguntas em que o resultado mudou. Ele diz se a melhora pode ter sido por acaso.

In [ ]:
# 9.2 Teste de McNemar (a implementar)

## 10. Análise de erros

**O que faz:** classifica cada erro das respostas com RAG em dois tipos:
- **Erro de recuperação:** o trecho com a resposta não estava entre os 4 recuperados.
- **Erro de geração:** o trecho estava lá, mas o modelo respondeu errado.

Também lista as respostas sem RAG que usaram o critério antigo de 2020.
**Por que:** responde à **P3** e mostra onde vale a pena melhorar: na busca ou no modelo.

In [ ]:
# 10. Análise de erros (a implementar)

## 11. Resultados

**O que faz:** reúne as tabelas e os gráficos finais (acurácia, Recall@4, kappa, McNemar, latência e custo) e grava os arquivos de saída que acompanham a entrega.
**Por que:** são os números e figuras que vão para o artigo e para a apresentação, todos gerados pelo próprio notebook.

In [ ]:
# 11. Tabelas, gráficos e exportação (a implementar)

## Limitações conhecidas

- O juiz não é especialista em saúde; a conferência humana (kappa) e a validação das perguntas por especialista reduzem esse risco.
- O mesmo modelo (Claude Opus 5.5) gerou as perguntas e julga as respostas.
- As perguntas não têm erros de digitação nem gírias; a robustez a isso não foi avaliada.
- Os modelos avaliados são pequenos (até 7B parâmetros); os resultados não se aplicam diretamente a modelos maiores.
- Com 50 perguntas, só diferenças grandes entre as condições são detectáveis estatisticamente.